In [1]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split as ttt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, layers

In [2]:
x = tf.random.normal((20000, 30))
y = tf.random.uniform((20000, 1), minval=0, maxval=2, dtype=tf.int32)

I0000 00:00:1790066879.341305      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790066879.343639      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [3]:
# scaling the dataset
x = np.array(x)
y = np.array(y)
x = x.astype('float32') / 255.0
print(type(x))
print(x.shape)

<class 'numpy.ndarray'>
(20000, 30)


In [4]:
x_train, x_test, y_train, y_test = ttt(x, y, test_size=0.3, random_state=42)

In [5]:
print(x_train.shape)
print(x_train[0].shape)
print(x_train[0])

(14000, 30)
(30,)
[-0.0024009   0.00466929 -0.00225322 -0.0031024   0.00454695  0.00205591
  0.00626881  0.00078177 -0.00627692 -0.00273168  0.00690443 -0.00183839
  0.01031133  0.00318898  0.00627649 -0.00203359  0.00233948 -0.01006962
  0.00214433  0.00220266 -0.00204114  0.00793389 -0.00333785  0.00291295
 -0.00545535  0.00571689 -0.0008895   0.00656234  0.00135648  0.00149048]


In [6]:
print(y_train.shape)
temp = pd.DataFrame(y_train)
print(temp[0].unique())

(14000, 1)
[1 0]


In [7]:
temp

,0
0,1
1,1
2,1
3,0
4,0
...,...
13995,0
13996,0
13997,0
13998,0


In [10]:
from tensorflow.keras.callbacks import EarlyStopping

In [14]:

# STEP 1
model = keras.Sequential([
    layers.Dense(25, activation='relu', kernel_initializer='he_normal'),
    layers.Dropout(0.15),
    layers.Dense(60, activation='relu', kernel_initializer='he_normal'),
    layers.Dropout(0.2),
    layers.Dense(150, activation='relu', kernel_initializer='he_normal'),
    layers.Dropout(0.2),
    layers.Dense(80, activation='relu', kernel_initializer='he_normal'),
    layers.Dense(40, activation='relu', kernel_initializer='he_normal'),
    layers.Dense(10, activation='relu', kernel_initializer='he_normal'),

    # output layer
    layers.Dense(1, activation='sigmoid', kernel_initializer='glorot_normal')
    
    
])


# STEP 2

model.compile(
    loss = keras.losses.BinaryCrossentropy(),
    optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    metrics = ['accuracy']
)


# STEP 3

early_stop = EarlyStopping(
    monitor = 'val_accuracy',
    patience = 5,
    restore_best_weights = True
)


# STEP 4

model.fit(
    x_train, y_train,
    batch_size = 128,
    epochs = 15,
    verbose = 1,
    validation_split = 0.3,
    callbacks = [early_stop]
)


# STEP 5

model.evaluate(
    x_test, y_test,
    batch_size=128,
    verbose = True
)


Epoch 1/15
77/77 ━━━━━━━━━━━━━━━━━━━━ 6s 40ms/step - accuracy: 0.4982 - loss: 0.6933 - val_accuracy: 0.4990 - val_loss: 0.6933
Epoch 2/15
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4955 - loss: 0.6931 - val_accuracy: 0.4990 - val_loss: 0.6934
Epoch 3/15
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5068 - loss: 0.6933 - val_accuracy: 0.4990 - val_loss: 0.6932
Epoch 4/15
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5066 - loss: 0.6931 - val_accuracy: 0.4990 - val_loss: 0.6933
Epoch 5/15
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5068 - loss: 0.6931 - val_accuracy: 0.4990 - val_loss: 0.6932
Epoch 6/15
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5069 - loss: 0.6931 - val_accuracy: 0.4990 - val_loss: 0.6932
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5040 - loss: 0.6932


[0.6931768655776978, 0.5040000081062317]